In [13]:
import pandas as pd 
import numpy as np
import sys

sys.path.append(r"D:\School\IITD\General\GBM")

gravendeel = pd.read_csv('D:/School/IITD/General/GBM/gravendeel/gravendeel_normalized.csv', index_col=0)
print(gravendeel.shape)
print(gravendeel.index[:5])
print(gravendeel.columns[:3].tolist())

(54675, 284)
Index(['1007_s_at', '1053_at', '117_at', '121_at', '1255_g_at'], dtype='object')
['GSM405200.CEL', 'GSM405201.CEL', 'GSM405202.CEL']


In [2]:
# Analysis in R gives 57147 probes and 22168 genes
# Loading probe mapping:

#probe_map = pd.read_csv('D:/School/IITD/General/GBM/gravendeel/probe_gene_mapping.csv')
#print(probe_map.head())
#print(probe_map.columns.tolist())

# First column is irrelevant, can be discarded:
probe_map = pd.read_csv('D:/School/IITD/General/GBM/gravendeel/probe_gene_mapping.csv', 
                        index_col=0)  # treats Unnamed:0 (the first column) as index

# Drop rows with no gene symbol
probe_map = probe_map.dropna(subset=['SYMBOL'])

#print(probe_map.shape)
#print(probe_map.head())

# After cleaning only 47119 probes actually mapped to genes

In [3]:
# Collapsing to one probe (one expression value) per gene based on max mean expression
# This is done to make the data compatible with MINER since we need only one expression value per gene per patient


# Add gene symbols to expression matrix
gravendeel_mapped = gravendeel.copy()
gravendeel_mapped.index.name = 'PROBEID'
gravendeel_mapped = gravendeel_mapped.reset_index()

# Merge with probe mapping
gravendeel_mapped = gravendeel_mapped.merge(probe_map, on='PROBEID', how='inner')

print(f"After merge: {gravendeel_mapped.shape}")

# Calculate mean expression per probe across all samples
sample_cols = [col for col in gravendeel_mapped.columns if col.startswith('GSM')]
gravendeel_mapped['mean_expr'] = gravendeel_mapped[sample_cols].mean(axis=1)

# Keep probe with highest mean expression per gene
gravendeel_mapped = gravendeel_mapped.loc[gravendeel_mapped.groupby('SYMBOL')['mean_expr'].idxmax()]

# Set gene symbol as index, keep only sample columns
gravendeel_gene = gravendeel_mapped.set_index('SYMBOL')[sample_cols]

#print(f"After collapsing to genes: {gravendeel_gene.shape}")
#print(gravendeel_gene.index[:5].tolist())

# After collapsing to 1 per gene: 22167 genes * 284 patients

After merge: (47119, 286)


In [5]:
import GEOparse as gp
gse = gp.get_GEO("GSE16011", destdir="D:/School/IITD/General/GBM/gravendeel")

20-May-2026 16:33:02 DEBUG utils - Directory D:/School/IITD/General/GBM/gravendeel already exists. Skipping.
20-May-2026 16:33:02 INFO GEOparse - File already exist: using local version.
20-May-2026 16:33:02 INFO GEOparse - Parsing D:/School/IITD/General/GBM/gravendeel\GSE16011_family.soft.gz: 
20-May-2026 16:33:02 DEBUG GEOparse - DATABASE: GeoMiame
20-May-2026 16:33:02 DEBUG GEOparse - SERIES: GSE16011
20-May-2026 16:33:02 DEBUG GEOparse - PLATFORM: GPL8542
20-May-2026 16:33:03 DEBUG GEOparse - SAMPLE: GSM405200
20-May-2026 16:33:03 DEBUG GEOparse - SAMPLE: GSM405201
20-May-2026 16:33:03 DEBUG GEOparse - SAMPLE: GSM405202
20-May-2026 16:33:03 DEBUG GEOparse - SAMPLE: GSM405203
20-May-2026 16:33:03 DEBUG GEOparse - SAMPLE: GSM405204
20-May-2026 16:33:03 DEBUG GEOparse - SAMPLE: GSM405205
20-May-2026 16:33:03 DEBUG GEOparse - SAMPLE: GSM405206
20-May-2026 16:33:03 DEBUG GEOparse - SAMPLE: GSM405207
20-May-2026 16:33:03 DEBUG GEOparse - SAMPLE: GSM405208
20-May-2026 16:33:03 DEBUG GEOpa

In [6]:
# Get sample metadata
phenodata = []
for gsm_id, gsm in gse.gsms.items():
    metadata = gsm.metadata
    phenodata.append({
        'gsm_id': gsm_id,
        'title': metadata.get('title', [None])[0],
        'characteristics': metadata.get('characteristics_ch1', [None])
    })

pheno_df = pd.DataFrame(phenodata)
print(pheno_df.shape)
print(pheno_df.head())

(284, 3)
      gsm_id      title                                    characteristics
0  GSM405200  control 7                [tissue: brain, histology: control]
1  GSM405201   glioma 8  [tissue: brain, gender: Female, histology: OD ...
2  GSM405202   glioma 9  [tissue: brain, gender: Male, histology: OD (g...
3  GSM405203  glioma 11  [tissue: brain, gender: Male, histology: OD (g...
4  GSM405204  glioma 13  [tissue: brain, gender: Male, histology: OD (g...


In [7]:
# Extract histology from characteristics
pheno_df['histology'] = pheno_df['characteristics'].apply(
    lambda x: [item for item in x if 'histology' in item][0].replace('histology: ', '')
    if any('histology' in item for item in x) else None
)

# Filter to glioma grades II-IV
grades_to_keep = ['OD (grade II)', 'OD (grade III)', 'GBM (grade IV)', 
                  'OA (grade II)', 'OA (grade III)', 'A (grade II)', 
                  'A (grade III)']

pheno_filtered = pheno_df[pheno_df['histology'].isin(grades_to_keep)]

# Total samples: 284
# After filtering to grades II-IV: 268

#print(f"Total samples: {len(pheno_df)}")
#print(f"After filtering to grades II-IV: {len(pheno_filtered)}")
#print(pheno_filtered['histology'].value_counts())

# Get GSM IDs for filtered samples
filtered_gsm_ids = pheno_filtered['gsm_id'].tolist()

# Match to expression matrix columns (which have .CEL suffix)
filtered_cols = [col for col in gravendeel_gene.columns 
                 if col.replace('.CEL', '') in filtered_gsm_ids]

gravendeel_filtered = gravendeel_gene[filtered_cols]

# Clean column names — remove .CEL suffix
gravendeel_filtered.columns = [col.replace('.CEL', '') for col in gravendeel_filtered.columns]

#print(f"Expression matrix after filtering: {gravendeel_filtered.shape}")
#print(gravendeel_filtered.columns[:3].tolist())

# Filtered shape of expr matrix is 22167 genes * 262 patients

In [8]:
# Z-scoring the data:

from scipy import stats

# Z-score per gene across all samples

gravendeel_zscore = pd.DataFrame(
    stats.zscore(gravendeel_filtered.values, axis=1),
    index=gravendeel_filtered.index,
    columns=gravendeel_filtered.columns
)

#print(f"Shape: {gravendeel_zscore.shape}")
#print(f"NaN check: {gravendeel_zscore.isna().sum().sum()}")
#print(gravendeel_zscore.iloc[:3, :3])

# Shape is still 22167 genes * 262 patients and 0 NaN values

In [9]:
# Loading expr_clean from gbm_model.ipynb

import pickle
with open(r"D:\School\IITD\General\GBM\saved_data\expr_clean.pkl", 'rb') as f:
    expr_clean= pickle.load(f)
    
#print(expr_clean.shape)
# Working fine: should be 13.7k * 437

In [10]:
# Finding common genes with TCGA:

common_genes_grav = gravendeel_zscore.index.intersection(expr_clean.index)
gravendeel_common = gravendeel_zscore.loc[common_genes_grav]
#print(len(gravendeel_common))

# Genes in Gravendeel: 22167
# Genes in TCGA (expr_clean): 13741
# Common genes: 12680
# Gravendeel filtered to common genes: (12680, 262)

In [11]:
import pickle
from sklearn.decomposition import PCA
from lifelines import CoxPHFitter
from scipy import stats

# Load from TCGA notebook

with open(r"D:\School\IITD\General\GBM\saved_data\validated.pkl", 'rb') as f:
    validated = pickle.load(f)

with open(r"D:\School\IITD\General\GBM\saved_data\eigengenes.pkl", 'rb') as f:
    eigengenes = pickle.load(f)

with open(r"D:\School\IITD\General\GBM\saved_data\cox_results.pkl", 'rb') as f:
    cox_results = pickle.load(f)

print("All loaded successfully")
print(f"expr_clean: {expr_clean.shape}")
print(f"Validated regulons: {len(validated)}")
print(f"Eigengenes: {len(eigengenes)}")
print(f"Cox results: {len(cox_results)}")

All loaded successfully
expr_clean: (13741, 437)
Validated regulons: 899
Eigengenes: 899
Cox results: 899


In [17]:
# Validate all 505 regulons in Gravendeel:
from utils import validate_coexpression, get_eigengene, cox_regulon

grav_coexp_validated = []
for idx, cluster in enumerate(validated):
    
    if idx % 100 == 0:
        print(f"Validating {idx}/{len(validated)}...")
        
    if validate_coexpression(cluster, gravendeel_common, min_genes = 3):
        grav_coexp_validated.append(idx)

        # Validated regulons from gbm_model are now tested on Gravendeel data
        # Their index (position within validated) is stored in a list

#print(f"Regulons passing coexpression in Gravendeel: {len(grav_coexp_validated)}")
# 630/899 were validated

Validating 0/899...
Validating 100/899...
Validating 200/899...
Validating 300/899...
Validating 400/899...
Validating 500/899...
Validating 600/899...
Validating 700/899...
Validating 800/899...


In [16]:
# Now to validate survival model
# However Gravendeel metadata does not have survival data
# This is instead in supplementary tables, which are PDFs

import pdfplumber

pdf_path = r"D:\School\IITD\General\GBM\gravendeel\00085472can092307-sup-stabs_1-6.pdf"
survival_rows = []

with pdfplumber.open(pdf_path) as pdf:
     for i, page in enumerate(pdf.pages[:9]):  # first 9 pages have survival data
        table = page.extract_table()
        if table:
            for row in table:
                try:
                    db_no = int(row[0])          # Database number
                    status = row[8]               # Alive/Dead/Lost
                    years_str = row[9]            # Survival years
                    
                    if years_str and years_str.strip():
                        years = float(years_str.replace(',', '.'))
                    else:
                        years = None
                    
                    survival_rows.append({
                        'db_no': db_no,
                        'status': status,
                        'OS_YEARS': years
                    })
                except (ValueError, TypeError, IndexError):
                    continue  # Skip header rows and malformed rows


surv_raw_df = pd.DataFrame(survival_rows)
# Map db_no to gsm_id and create final survival dataframe
# Extract db number from title for all samples
db_to_gsm = {}
for _, row in pheno_df.iterrows():
    title = row['title']
    parts = title.split()
    if len(parts) == 2:
        try:
            db_no = int(parts[1])
            db_to_gsm[db_no] = row['gsm_id']
        except:
            continue

#print(f"db_to_gsm length: {len(db_to_gsm)}")
#print(list(db_to_gsm.items())[:5])

surv_rows = []
for _, row in surv_raw_df.iterrows():
    gsm_id = db_to_gsm.get(row['db_no'])
    if gsm_id and row['OS_YEARS'] is not None:
        surv_rows.append({
            'gsm_id': gsm_id,
            'OS_MONTHS': row['OS_YEARS'] * 12,
            'OS_STATUS': 1 if row['status'] == 'Dead' else 0
        })

grav_survival = pd.DataFrame(surv_rows).set_index('gsm_id')
print(grav_survival.shape)
print(grav_survival.head())
grav_survival = pd.DataFrame(surv_rows).set_index('gsm_id')
grav_survival = grav_survival.dropna()

# Survival data: (262, 2) (dropped incomplete data)
# Deaths: 233
# Alive/Censored: 29
# This matches the filtered expression matrix (gravendeel_common) which has 7393 genes across 262 patients

# Align survival to filtered expression patients
common_patients_grav = grav_survival.index.intersection(gravendeel_common.columns)
grav_survival_aligned = grav_survival.loc[common_patients_grav]
gravendeel_aligned = gravendeel_common[common_patients_grav]

# 252 common patients between gravendeel_common and our survival data
# gravendeel_aligned is a 12680 * 252 (genes * patients) TUPLE

# Finally extracted for 275 patients (9 may be controls/ missing data)

# Print first few rows of the PDF table to inspect structure

(273, 2)
           OS_MONTHS  OS_STATUS
gsm_id                         
GSM405200        NaN          0
GSM405201     117.84          1
GSM405202     140.04          1
GSM405203     107.04          1
GSM405204     103.08          1


In [18]:
# How many genes from first cluster are in Gravendeel?
cluster = validated[0]
in_grav = [g for g in cluster if g in gravendeel_aligned.index]
print(f"Cluster size: {len(cluster)}")
print(f"Genes in Gravendeel: {len(in_grav)}")
print(f"Cluster genes: {cluster[:5]}")

Cluster size: 116
Genes in Gravendeel: 86
Cluster genes: ['OR52M1', 'ALPI', 'SLC34A1', 'HMGCS2', 'KLK3']


In [19]:
# Eigengene computation:

grav_eigen_genes = {}
for idx, cluster in enumerate(validated):
    if idx % 100 == 0:
        print(f"Computing eigen gene {idx}/{len(validated)}...")
    
    cluster_in_grav = [g for g in cluster if g in gravendeel_aligned.index]
    
    if len(cluster_in_grav) < 3:
        continue
    
    eigen = get_eigengene(cluster_in_grav, gravendeel_aligned)
    
    if eigen is not None:
        grav_eigen_genes[idx] = eigen

print(f"Eigen genes computed: {len(grav_eigen_genes)}")

Computing eigen gene 0/899...
Computing eigen gene 100/899...
Computing eigen gene 200/899...
Computing eigen gene 300/899...
Computing eigen gene 400/899...
Computing eigen gene 500/899...
Computing eigen gene 600/899...
Computing eigen gene 700/899...
Computing eigen gene 800/899...
Eigen genes computed: 895


In [21]:
grav_cox_results = {}

for idx, eigen in grav_eigen_genes.items():
    try:
        df = pd.DataFrame({
            'eigen': eigen,
            'OS_MONTHS': grav_survival_aligned['OS_MONTHS'],
            'OS_STATUS': grav_survival_aligned['OS_STATUS']
        }).dropna()

        if df.shape[0] < 10:
            continue

        cph = CoxPHFitter()
        cph.fit(df, duration_col='OS_MONTHS', event_col='OS_STATUS')

        summary = cph.summary.loc['eigen']
        grav_cox_results[idx] = {
            'hr': summary['exp(coef)'],
            'pval': summary['p']
        }

    except Exception as e:
        continue

print(f"Gravendeel Cox done: {len(grav_cox_results)} regulons tested")
sig = {k: v for k, v in grav_cox_results.items() if v['pval'] <= 0.05}
print(f"Significant (p≤0.05): {len(sig)}")

Gravendeel Cox done: 895 regulons tested
Significant (p≤0.05): 630


In [22]:
# Final set of disease-relevant regulons i.e overlap between TCGA and Gravendeel

tcga_sig = set(k for k, v in cox_results.items() if v['pval'] <= 0.05)
grav_sig = set(k for k, v in grav_cox_results.items() if v['pval'] <= 0.05)

overlap = tcga_sig & grav_sig
print(f"TCGA significant: {len(tcga_sig)}")
print(f"Gravendeel significant: {len(grav_sig)}")
print(f"Cross-validated overlap: {len(overlap)}")

TCGA significant: 181
Gravendeel significant: 630
Cross-validated overlap: 148


In [23]:
# Saving for later:

import pickle

# Save the validated set
disease_regulons = list(overlap)
print(f"Disease-relevant regulons: {len(disease_regulons)}")

with open(r"D:\School\IITD\General\GBM\saved_data\disease_regulons.pkl", 'wb') as f:
    pickle.dump(disease_regulons, f)

with open(r"D:\School\IITD\General\GBM\saved_data\grav_cox_results.pkl", 'wb') as f:
    pickle.dump(grav_cox_results, f)

print("Saved.")

Disease-relevant regulons: 148
Saved.


In [25]:
with open(r"D:\School\IITD\General\GBM\saved_data\gravendeel_aligned.pkl", 'wb') as f:
    pickle.dump(gravendeel_aligned, f)

with open(r"D:\School\IITD\General\GBM\saved_data\grav_survival_aligned.pkl", 'wb') as f:
    pickle.dump(grav_survival_aligned, f)

with open(r"D:\School\IITD\General\GBM\saved_data\grav_cox_results.pkl", 'wb') as f:
    pickle.dump(grav_cox_results, f)

